In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models  #type:ignore
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import mean_squared_error
from tensorflow.keras import regularizers

2026-04-30 11:06:44.354021: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777547204.540069      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777547204.588954      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777547205.036281      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777547205.036319      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777547205.036322      23 computation_placer.cc:177] computation placer alr

In [2]:
cols = ['unit', 'cycle', 'os1', 'os2', 'os3'] + [f's{i}' for i in range(1, 22)]

def load_data(path):
    return pd.read_csv(path, sep=' +', header=None,
                       usecols=range(26), names=cols, engine='python')

train = load_data('/kaggle/input/datasets/bishals098/nasa-turbofan-engine-degradation-simulation/train_FD004.txt')
test  = load_data('/kaggle/input/datasets/bishals098/nasa-turbofan-engine-degradation-simulation/test_FD004.txt')
rul   = pd.read_csv('/kaggle/input/datasets/bishals098/nasa-turbofan-engine-degradation-simulation/RUL_FD004.txt', header=None, names=['rul'])

In [3]:
train.head()

,unit,cycle,os1,os2,os3,s1,s2,s3,s4,s5,...,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,...,129.78,2387.99,8074.83,9.3335,0.02,330,2212,100.00,10.62,6.3670
1,1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,...,312.59,2387.73,8046.13,9.1913,0.02,361,2324,100.00,24.37,14.6552
2,1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,...,129.62,2387.97,8066.62,9.4007,0.02,329,2212,100.00,10.48,6.4213
3,1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,...,129.80,2388.02,8076.05,9.3369,0.02,328,2212,100.00,10.54,6.4176
4,1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,...,164.11,2028.08,7865.80,10.8366,0.02,305,1915,84.93,14.03,8.6754


In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61249 entries, 0 to 61248
Data columns (total 26 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   unit    61249 non-null  int64  
 1   cycle   61249 non-null  int64  
 2   os1     61249 non-null  float64
 3   os2     61249 non-null  float64
 4   os3     61249 non-null  float64
 5   s1      61249 non-null  float64
 6   s2      61249 non-null  float64
 7   s3      61249 non-null  float64
 8   s4      61249 non-null  float64
 9   s5      61249 non-null  float64
 10  s6      61249 non-null  float64
 11  s7      61249 non-null  float64
 12  s8      61249 non-null  float64
 13  s9      61249 non-null  float64
 14  s10     61249 non-null  float64
 15  s11     61249 non-null  float64
 16  s12     61249 non-null  float64
 17  s13     61249 non-null  float64
 18  s14     61249 non-null  float64
 19  s15     61249 non-null  float64
 20  s16     61249 non-null  float64
 21  s17     61249 non-null  int64  
 22

In [5]:
DROP_SENSORS = ['s1','s5','s6','s8','s10','s13','s15','s16','s18','s19']
KEEP_SENSORS = ['s2','s3','s4','s7','s9','s11','s12','s14','s17','s20','s21']

train.drop(columns=DROP_SENSORS, inplace=True)
test.drop(columns=DROP_SENSORS, inplace=True)

In [6]:
def add_rul(df):
    max_cycle = df.groupby('unit')['cycle'].transform('max')
    df['RUL'] = max_cycle - df['cycle']
    return df

train = add_rul(train)

last_cycles = test.groupby('unit')['cycle'].max().reset_index()
last_cycles['RUL'] = rul['rul'].values
test = test.merge(last_cycles[['unit','RUL']], on='unit', how='left')
test_last = test.groupby('unit').last().reset_index()

In [7]:
RUL_CLIP = 125
train['RUL'] = train['RUL'].clip(upper=RUL_CLIP)

In [8]:
SETTING_COLS = ['os1', 'os2', 'os3']
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
train['condition'] = kmeans.fit_predict(train[SETTING_COLS])
test['condition']  = kmeans.predict(test[SETTING_COLS])

In [9]:
train[KEEP_SENSORS] = train[KEEP_SENSORS].astype(np.float64)
test[KEEP_SENSORS]  = test[KEEP_SENSORS].astype(np.float64)

scalers = {}
for cond in range(6):
    scaler = MinMaxScaler()
    mask_tr = train['condition'] == cond
    mask_te = test['condition']  == cond
    train.loc[mask_tr, KEEP_SENSORS] = scaler.fit_transform(train.loc[mask_tr, KEEP_SENSORS])
    if mask_te.sum() > 0:
        test.loc[mask_te, KEEP_SENSORS] = scaler.transform(test.loc[mask_te, KEEP_SENSORS])
    scalers[cond] = scaler

In [10]:
print("=== RUL range (train should max at 125) ===")
print(f"Train RUL: min={train['RUL'].min()}, max={train['RUL'].max()}")
print(f"Test  RUL: min={test_last['RUL'].min()}, max={test_last['RUL'].max()}")

print("\n=== Sensor range (should be 0.0 to 1.0) ===")

train_min = train[KEEP_SENSORS].min().min()
train_max = train[KEEP_SENSORS].max().max()

test_min = test[KEEP_SENSORS].min().min()
test_max = test[KEEP_SENSORS].max().max()

print("Train:", round(train_min, 3), "to", round(train_max, 3))
print("Test: ", round(test_min, 3),  "to", round(test_max, 3))

=== RUL range (train should max at 125) ===
Train RUL: min=0, max=125
Test  RUL: min=6, max=195

=== Sensor range (should be 0.0 to 1.0) ===
Train: 0.0 to 1.0
Test:  -0.091 to 1.009


In [11]:
WINDOW   = 50
RUL_CLIP = 125

def build_sequences(df, feature_cols, window=WINDOW):
    X, y = [], []
    for _, engine_df in df.groupby('unit'):
        engine_df = engine_df.sort_values('cycle')
        data, labels = engine_df[feature_cols].values, engine_df['RUL'].values
        for i in range(len(data) - window + 1):
            X.append(data[i:i+window])
            y.append(labels[i+window-1])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

def build_last_sequences(df, feature_cols, window=WINDOW):
    X, y = [], []
    for _, engine_df in df.groupby('unit'):
        engine_df = engine_df.sort_values('cycle')
        data = engine_df[feature_cols].values
        if len(data) >= window:
            X.append(data[-window:])
        else:
            pad = np.zeros((window - len(data), len(feature_cols)))
            X.append(np.vstack([pad, data]))
        y.append(engine_df['RUL'].iloc[-1])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


In [12]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(train, groups=train['unit']))

X_train, y_train_raw = build_sequences(train.iloc[train_idx], KEEP_SENSORS, WINDOW)
X_val,   y_val_raw   = build_sequences(train.iloc[val_idx],   KEEP_SENSORS, WINDOW)
X_test,  y_test_raw  = build_last_sequences(test, KEEP_SENSORS, WINDOW)

y_train = (y_train_raw / RUL_CLIP).astype(np.float32)
y_val   = (y_val_raw   / RUL_CLIP).astype(np.float32)
y_test_clipped = np.clip(y_test_raw, 0, RUL_CLIP).astype(np.float32)

print("=== Data shapes ===")
print(f"Sequences rebuilt with window={WINDOW}")
print(f"X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}")
print(f"y_train: min={y_train.min():.3f}, max={y_train.max():.3f}  (should be 0.0–1.0)")
print(f"y_val:   min={y_val.min():.3f},   max={y_val.max():.3f}")
print(f"y_test_clipped (cycles): min={y_test_clipped.min()}, max={y_test_clipped.max()}")

=== Data shapes ===
Sequences rebuilt with window=50
X_train=(39543, 50, 11), X_val=(9505, 50, 11), X_test=(248, 50, 11)
y_train: min=0.000, max=1.000  (should be 0.0–1.0)
y_val:   min=0.000,   max=1.000
y_test_clipped (cycles): min=6.0, max=125.0


In [13]:
unique, counts = np.unique((y_train_raw // 10) * 10, return_counts=True)
print("RUL distribution BEFORE rebalance:")
for u, c in zip(unique, counts):
    bar = '█' * (c // 200)
    print(f"  RUL {int(u):3d}-{int(u+9):3d}: {c:5d}  {bar}")

RUL distribution BEFORE rebalance:
  RUL   0-  9:  1990  █████████
  RUL  10- 19:  1990  █████████
  RUL  20- 29:  1990  █████████
  RUL  30- 39:  1990  █████████
  RUL  40- 49:  1990  █████████
  RUL  50- 59:  1990  █████████
  RUL  60- 69:  1990  █████████
  RUL  70- 79:  1989  █████████
  RUL  80- 89:  1980  █████████
  RUL  90- 99:  1960  █████████
  RUL 100-109:  1910  █████████
  RUL 110-119:  1830  █████████
  RUL 120-129: 15944  ███████████████████████████████████████████████████████████████████████████████


In [14]:
def rebalance_sequences(X, y_raw, bucket_size=10, max_per_bucket=800):
    buckets  = (y_raw // bucket_size).astype(int)
    keep_idx = []
    for b in np.unique(buckets):
        idx = np.where(buckets == b)[0]
        if len(idx) > max_per_bucket:
            idx = np.random.choice(idx, max_per_bucket, replace=False)
        keep_idx.append(idx)
    keep_idx = np.concatenate(keep_idx)
    np.random.shuffle(keep_idx)
    return X[keep_idx], y_raw[keep_idx]

In [15]:
np.random.seed(42)
X_train, y_train_raw = rebalance_sequences(X_train, y_train_raw, max_per_bucket=800)

unique, counts = np.unique((y_train_raw // 10) * 10, return_counts=True)
print(f"\nRUL distribution AFTER rebalance (total={len(X_train)}):")
for u, c in zip(unique, counts):
    bar = '█' * (c // 50)
    print(f"  RUL {int(u):3d}-{int(u+9):3d}: {c:5d}  {bar}")

# NOW normalize
y_train = (y_train_raw / RUL_CLIP).astype(np.float32)
sample_weights = (1.0 + 1.5 * (y_train_raw / RUL_CLIP)).astype(np.float32)


RUL distribution AFTER rebalance (total=10400):
  RUL   0-  9:   800  ████████████████
  RUL  10- 19:   800  ████████████████
  RUL  20- 29:   800  ████████████████
  RUL  30- 39:   800  ████████████████
  RUL  40- 49:   800  ████████████████
  RUL  50- 59:   800  ████████████████
  RUL  60- 69:   800  ████████████████
  RUL  70- 79:   800  ████████████████
  RUL  80- 89:   800  ████████████████
  RUL  90- 99:   800  ████████████████
  RUL 100-109:   800  ████████████████
  RUL 110-119:   800  ████████████████
  RUL 120-129:   800  ████████████████


In [16]:
def build_gru(window, n_features):
    inp = tf.keras.Input(shape=(window, n_features))
    x = layers.GRU(128, return_sequences=True)(inp)
    x = layers.Dropout(0.2)(x)
    x = layers.GRU(64, return_sequences=True)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.GRU(32)(x)
    x = layers.Dropout(0.15)(x)
    x = layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    return models.Model(inp, out)

In [17]:
model = build_gru(WINDOW, len(KEEP_SENSORS))
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss='mse', 
    metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse')]
)
model.summary()

I0000 00:00:1777547232.759557      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 50, 11)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 50, 128)        │        54,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 50, 64)         │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 50, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 102,401 (400.00 KB)

 Trainable params: 102,401 (400.00 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=10,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=5, min_lr=1e-5, verbose=1
    )
]

sample_weights = (1.0 + 1.5 * (y_train_raw / RUL_CLIP)).astype(np.float32)

history = model.fit(
    X_train, y_train,       
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=512,
    sample_weight=sample_weights,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/100


I0000 00:00:1777547237.518493      69 cuda_dnn.cc:529] Loaded cuDNN version 91002


21/21 ━━━━━━━━━━━━━━━━━━━━ 6s 56ms/step - loss: 0.1614 - rmse: 0.3039 - val_loss: 0.1150 - val_rmse: 0.3312 - learning_rate: 3.0000e-04
Epoch 2/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - loss: 0.1311 - rmse: 0.2785 - val_loss: 0.0787 - val_rmse: 0.2710 - learning_rate: 3.0000e-04
Epoch 3/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - loss: 0.0801 - rmse: 0.2119 - val_loss: 0.0548 - val_rmse: 0.2227 - learning_rate: 3.0000e-04
Epoch 4/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - loss: 0.0610 - rmse: 0.1759 - val_loss: 0.0454 - val_rmse: 0.2006 - learning_rate: 3.0000e-04
Epoch 5/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - loss: 0.0557 - rmse: 0.1667 - val_loss: 0.0430 - val_rmse: 0.1947 - learning_rate: 3.0000e-04
Epoch 6/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - loss: 0.0545 - rmse: 0.1646 - val_loss: 0.0438 - val_rmse: 0.1969 - learning_rate: 3.0000e-04
Epoch 7/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - loss: 0.0545 - rmse: 0.1643 - val_loss: 0.0434 - val_rmse: 0.1961 - le

In [19]:
def nasa_score(y_true, y_pred):
    d = y_pred.flatten() - y_true.flatten()
    return float(np.sum(np.where(d < 0, np.exp(-d/13)-1, np.exp(d/10)-1)))

val_preds_cycles  = model.predict(X_val).flatten()  * RUL_CLIP
test_preds_cycles = model.predict(X_test).flatten() * RUL_CLIP
test_preds_cycles = np.clip(test_preds_cycles, 0, RUL_CLIP)

y_val_cycles = y_val_raw  # unclipped, in cycles — val engines are from training data so max=125

rmse_val  = np.sqrt(mean_squared_error(y_val_cycles,  val_preds_cycles))
rmse_test = np.sqrt(mean_squared_error(y_test_clipped, test_preds_cycles))

print(f"\nVal RMSE:   {rmse_val:.2f}")
print(f"Test RMSE:  {rmse_test:.2f}")
print(f"NASA Score: {nasa_score(y_test_clipped, test_preds_cycles):.1f}")
print(f"Pred range: {test_preds_cycles.min():.1f} – {test_preds_cycles.max():.1f}")
print(f"True range: {y_test_clipped.min():.1f} – {y_test_clipped.max():.1f}")

298/298 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 

Val RMSE:   17.94
Test RMSE:  26.08
NASA Score: 25212.3
Pred range: 7.6 – 120.7
True range: 6.0 – 125.0


In [20]:
results = pd.DataFrame({
    'true_RUL': y_test_clipped,
    'pred_RUL': test_preds_cycles,
    'error':    test_preds_cycles - y_test_clipped,
    'abs_error': np.abs(test_preds_cycles - y_test_clipped)
})

In [21]:
print("\n=== Error by RUL bucket ===")
bins = [0, 30, 60, 90, 125]
labels = ['critical(0-30)', 'degrading(30-60)', 'mid(60-90)', 'healthy(90-125)']
results['bucket'] = pd.cut(results['true_RUL'], bins=bins, labels=labels)
print(results.groupby('bucket', observed=True)['abs_error'].agg(['mean','count']).round(2))


=== Error by RUL bucket ===
                       mean  count
bucket                            
critical(0-30)     4.790000     53
degrading(30-60)  13.940000     38
mid(60-90)        14.570000     40
healthy(90-125)   22.440001    117


In [22]:

results['critical_true'] = results['true_RUL'] < 30
results['critical_pred'] = results['pred_RUL'] < 30
print("\n=== Critical zone (RUL < 30) ===")
print(confusion_matrix(results['critical_true'], results['critical_pred']))
print(classification_report(results['critical_true'], results['critical_pred'],
                             target_names=['safe', 'critical']))


=== Critical zone (RUL < 30) ===
[[187   9]
 [  4  48]]
              precision    recall  f1-score   support

        safe       0.98      0.95      0.97       196
    critical       0.84      0.92      0.88        52

    accuracy                           0.95       248
   macro avg       0.91      0.94      0.92       248
weighted avg       0.95      0.95      0.95       248

